Ändra från normalfördelade

## Simulated dataset (patients × proteins)

To sanity-check our pipeline under controlled conditions, we simulate synthetic datasets with a known ground-truth signal structure. The simulator returns a protein matrix $X \in \mathbb{R}^{n \times p}$ (patients × proteins) and a binary ARDS label $y \in \{0,1\}^n$, together with a `truth` object containing the causal proteins and (optionally) interaction pairs.

### 1) Correlated proteins via latent pathways
We introduce $K$ latent pathway activities per patient:
$$
Z_{ik} \sim \mathcal{N}(0,1), \quad i=1,\dots,n,\; k=1,\dots,K.
$$
Each protein $j$ is assigned to one pathway block $b(j)$. A loading matrix $W \in \mathbb{R}^{K \times p}$ is constructed such that each protein loads on its assigned pathway with strength $\lambda$ (`block_strength`). Protein values are generated as:
$$
X = ZW + E, \qquad E_{ij} \sim \mathcal{N}(0,\sigma^2),
$$
where $\sigma$ (`noise_sd`) controls the signal-to-noise ratio.

### 2) Ground truth: main effects and/or interactions
We support three signal modes (`effect_mode`):

- **`main`**: select a set $S$ of $m$ signal proteins (`n_signal`) and assign coefficients $\beta_j$ (positive or negative).  
  $$
  \eta_i = \sum_{j \in S} \beta_j X_{ij} = X_{i,S}^\top \boldsymbol{\beta}.
  $$

- **`interaction_only`**: generate outcomes only from interaction terms (no main effects):  
  $$
  \eta_i = \sum_{t=1}^{T} \gamma_t\, X_{i,j_1(t)}X_{i,j_2(t)}.
  $$

- **`main_plus_interaction`**: combine both main effects and interactions:
  $$
  \eta_i = X_{i,S}^\top \boldsymbol{\beta} + \sum_{t=1}^{T} \gamma_t\, X_{i,j_1(t)}X_{i,j_2(t)}.
  $$
Interaction pairs $(j_1, j_2)$ are sampled uniquely (unordered) either from the signal set $S$ or from all proteins, controlled by `interaction_from_signals`.

### 3) Prevalence calibration and label sampling
We map the linear predictor to probabilities with a logistic link:
$$
\pi_i = \sigma(\alpha + \eta_i) = \frac{1}{1 + e^{-(\alpha + \eta_i)}}.
$$
The intercept $\alpha$ is iteratively adjusted so that the expected prevalence matches the target `ards_rate`. Finally,
$$
y_i \sim \text{Bernoulli}(\pi_i).
$$

This setup allows direct evaluation of (i) predictive performance and (ii) feature recovery against the known ground truth, under controlled correlation and noise.



In [1]:
import numpy as np
import pandas as pd
from scipy.special import expit  # sigmoid

In [ ]:

# Patients x proteins
def simulate_patients(
    n=250, p=300, K=10,
    ards_rate=0.2,

    # signal/main effects
    n_signal=8,
    beta_signal=0.9,
    beta_sd=0.6,

    # interaction effects
    n_interactions=0,
    gamma_signal=0.5,
    gamma_sd=0.4,
    interaction_from_signals=True,

    # correlation/noise
    block_strength=1.0,
    noise_sd=1.0,

    # mode: "main", "interaction_only", "main_plus_interaction"
    effect_mode="main",

    seed=0
):
    rng = np.random.default_rng(seed)

    if effect_mode not in {"main", "interaction_only", "main_plus_interaction"}:
        raise ValueError("effect_mode must be 'main', 'interaction_only', or 'main_plus_interaction'")

    # 1) latent pathways per patient
    Z = rng.normal(0, 1, size=(n, K))  # n patients x K pathways

    # 2) assign proteins to blocks (roughly equal)
    block_id = np.repeat(np.arange(K), p // K)
    if len(block_id) < p:
        block_id = np.concatenate([block_id, rng.integers(0, K, size=p - len(block_id))])
    rng.shuffle(block_id)

    # 3) loading matrix W: each protein loads on its block
    W = np.zeros((K, p))
    for j in range(p):
        W[block_id[j], j] = block_strength

    # 4) protein values
    X = Z @ W + rng.normal(0, noise_sd, size=(n, p))

    # bookkeeping defaults
    signal_idx = None
    beta_vec = None
    interaction_pairs = []
    gamma_vec = None

######## assure that some pathways have no link to ARDS (subset V of pathways -> subset of proteins S of subset V)

    # build linear predictor 
    eta = np.zeros(n)

    # 5) main effects (optional depending on mode)
    if effect_mode in {"main", "main_plus_interaction"}:
        if n_signal <= 0:
            raise ValueError("n_signal must be >= 1 for effect_mode 'main' or 'main_plus_interaction'")
        signal_idx = rng.choice(p, size=n_signal, replace=False)

        beta_vec = rng.normal(loc=0.0, scale=beta_sd, size=n_signal)  # +/- allowed
        beta_vec *= beta_signal / (np.std(beta_vec) + 1e-12)          # scale overall strength

        eta += X[:, signal_idx] @ beta_vec

    # 6) interaction effects (optional depending on mode)
    if effect_mode in {"interaction_only", "main_plus_interaction"}:
        if n_interactions <= 0:
            raise ValueError("n_interactions must be >= 1 for effect_mode 'interaction_only' or 'main_plus_interaction'")

        # choose pool for interactions
        if interaction_from_signals:
            if effect_mode == "interaction_only":
                # no main signals in this mode -> fall back to all proteins
                pool = np.arange(p)
            else:
                # main_plus_interaction: use the main signal proteins
                if signal_idx is None or len(signal_idx) < 2:
                    raise ValueError("Need at least 2 signal proteins to draw interactions from signals.")
                pool = signal_idx
        else:
            pool = np.arange(p)

        # maximum number of unique unordered pairs
        max_pairs = len(pool) * (len(pool) - 1) // 2
        if n_interactions > max_pairs:
            raise ValueError("Too many interaction pairs requested for the chosen pool.")

        # sample unique unordered pairs
        pair_set = set()
        while len(pair_set) < n_interactions:
            j1, j2 = rng.choice(pool, size=2, replace=False)
            pair_set.add(tuple(sorted((int(j1), int(j2)))))
        interaction_pairs = list(pair_set)

        gamma_vec = rng.normal(loc=0.0, scale=gamma_sd, size=n_interactions)  # +/- allowed
        gamma_vec *= gamma_signal / (np.std(gamma_vec) + 1e-12)               # scale overall strength

        for t, (j1, j2) in enumerate(interaction_pairs):
            eta += gamma_vec[t] * (X[:, j1] * X[:, j2])

    # 7) choose intercept to match target prevalence
    b0 = 0.0
    for _ in range(50):
        probs = expit(b0 + eta)
        diff = probs.mean() - ards_rate
        b0 -= diff * 2.0  # step size (simple fixed-point / gradient-like update)

    probs = expit(b0 + eta)
    y = rng.binomial(1, probs)

    # return as DataFrame
    X_df = pd.DataFrame(X, columns=[f"prot_{j:04d}" for j in range(p)])
    y_s = pd.Series(y.astype(int), name="ards")

    truth = {
        "effect_mode": effect_mode,

        "signal_idx": signal_idx,
        "signal_proteins": None if signal_idx is None else [X_df.columns[i] for i in signal_idx],
        "beta_vec": beta_vec,

        "interaction_pairs": interaction_pairs,
        "interaction_pair_names": [(X_df.columns[j1], X_df.columns[j2]) for (j1, j2) in interaction_pairs],
        "gamma_vec": gamma_vec,

        "block_id": block_id,
        "seed": seed,
        "b0": float(b0),
        "prevalence": float(y_s.mean()),
        "params": dict(
            n=n, p=p, K=K,
            ards_rate=ards_rate,
            n_signal=n_signal, beta_signal=beta_signal, beta_sd=beta_sd,
            n_interactions=n_interactions, gamma_signal=gamma_signal, gamma_sd=gamma_sd,
            interaction_from_signals=interaction_from_signals,
            block_strength=block_strength, noise_sd=noise_sd,
            effect_mode=effect_mode
        )
    }

    return X_df, y_s, truth

In [ ]:
def _make_latent_pathways(rng, n, K):
    Z = rng.normal(0, 1, size=(n, K))
    return Z

# rows = patients, cols = pathways
Z = _make_latent_pathways(np.random.default_rng(0), 10, 5)
print(Z)


[[ 0.12573022 -0.13210486  0.64042265  0.10490012 -0.53566937]
 [ 0.36159505  1.30400005  0.94708096 -0.70373524 -1.26542147]
 [-0.62327446  0.04132598 -2.32503077 -0.21879166 -1.24591095]
 [-0.73226735 -0.54425898 -0.31630016  0.41163054  1.04251337]
 [-0.12853466  1.36646347 -0.66519467  0.35151007  0.90347018]
 [ 0.0940123  -0.74349925 -0.92172538 -0.45772583  0.22019512]
 [-1.00961818 -0.20917557 -0.15922501  0.54084558  0.21465912]
 [ 0.35537271 -0.65382861 -0.12961363  0.78397547  1.49343115]
 [-1.25906553  1.51392377  1.34587542  0.7813114   0.26445563]
 [-0.31392281  1.45802068  1.96025832  1.80163487  1.31510376]]


In [10]:
def _assign_blocks(rng, p, K):
    block_id = np.repeat(np.arange(K), p // K)
    if len(block_id) < p:
        block_id = np.concatenate([block_id, rng.integers(0, K, size=p - len(block_id))])
    rng.shuffle(block_id)
    return block_id

block_id = _assign_blocks(np.random.default_rng(0), 20, 5)
print(block_id)

[1 4 1 0 3 4 0 2 2 2 0 3 1 1 4 4 3 2 0 3]


In [ ]:
def _make_loading_matrix(block_id, K, p, block_strength):
    W = np.zeros((K, p), dtype=float)
    for j in range(p):
        W[block_id[j], j] = block_strength
    return W

W = _make_loading_matrix(block_id, K=5, p=20, block_strength=1.0)
print(W)

# protein column lies in pathway row

[[0. 0. 0. 1. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [1. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 1. 0. 0. 1.]
 [0. 1. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 0. 0. 0. 0.]]


In [12]:
def _make_proteins(rng, Z, W, noise_sd):
    n, _ = Z.shape
    _, p = W.shape
    X = Z @ W + rng.normal(0, noise_sd, size=(n, p))
    return X

proteins = _make_proteins(np.random.default_rng(0), Z, W, noise_sd=1.0)
print(proteins)

[[-0.00637464 -0.66777424  0.50831779  0.23063034 -0.43076926 -0.17407432
   1.42973027  1.58750361 -0.06331259 -0.62499882 -0.49754424  0.1462261
  -2.45713564 -0.35089653 -1.78158032 -1.26793673 -0.43935887  0.32412249
   0.53736076  1.14741349]
 [ 1.17546538  0.101042    0.63880537  0.71310513  0.19973495 -1.17140917
  -0.38190419  0.02535559  0.48935514  1.16727609 -0.64802313 -0.91291081
   1.14477504  1.84484563 -1.05076235 -0.91004876 -1.35756385  0.81746733
   1.14557052  0.78969591]
 [-1.21773955  0.26801283  1.3872014   0.15803694  0.04566397 -1.55983376
   0.83474622 -0.36477246 -0.5233959  -1.00992701 -0.26589405 -1.4271103
   0.03687185  0.69780091 -2.53427241 -0.85078889  0.21107203 -1.62898805
  -1.80739243 -0.88049424]
 [-0.98069423 -0.12728854  1.19510889 -1.22817808  0.74060017  0.78394082
   0.85120552  1.00406083  0.31705247 -2.51981004 -0.68023838  1.09531673
   0.45970259 -1.16216603  2.86452473 -0.2779176  -0.24989749  0.61874983
  -0.68321274  2.41402312]
 [ 1.5

In [13]:
def _sample_main_effects(rng, X, p, n_signal, beta_signal, beta_sd):
    signal_idx = rng.choice(p, size=n_signal, replace=False)
    beta_vec = rng.normal(loc=0.0, scale=beta_sd, size=n_signal)  # +/- allowed
    beta_vec *= beta_signal / (np.std(beta_vec) + 1e-12)          # normalize strength
    eta_main = X[:, signal_idx] @ beta_vec
    return signal_idx, beta_vec, eta_main

signal_idx, beta_vec, eta_main = _sample_main_effects(np.random.default_rng(0), proteins, p=20, n_signal=5, beta_signal=0.9, beta_sd=0.6)
print("Signal indices:", signal_idx)
print("Beta vector:", beta_vec)
print("Main effects:", eta_main)

Signal indices: [10  9  5  6 13]
Beta vector: [ 0.33417696  1.20512372  0.87526817 -0.65037423 -1.16947038]
Main effects: [-1.59132706 -1.74425114 -4.03017043 -1.77232778  1.40521544 -0.70033096
 -2.32451901  4.52823607  1.40953616  1.51034888]


In [14]:
def _sample_interactions(
    rng, X, p,
    n_interactions, gamma_signal, gamma_sd,
    pool
):
    # unique unordered pairs
    max_pairs = len(pool) * (len(pool) - 1) // 2
    if n_interactions > max_pairs:
        raise ValueError("Too many interaction pairs requested for the chosen pool.")

    pair_set = set()
    while len(pair_set) < n_interactions:
        j1, j2 = rng.choice(pool, size=2, replace=False)
        pair_set.add(tuple(sorted((int(j1), int(j2)))))
    interaction_pairs = list(pair_set)

    gamma_vec = rng.normal(loc=0.0, scale=gamma_sd, size=n_interactions)  # +/- allowed
    gamma_vec *= gamma_signal / (np.std(gamma_vec) + 1e-12)

    eta_int = np.zeros(X.shape[0], dtype=float)
    for t, (j1, j2) in enumerate(interaction_pairs):
        eta_int += gamma_vec[t] * (X[:, j1] * X[:, j2])

    return interaction_pairs, gamma_vec, eta_int

interaction_pairs, gamma_vec, eta_int = _sample_interactions(
    np.random.default_rng(0), proteins, p=20,
    n_interactions=3, gamma_signal=0.5, gamma_sd=0.4,
    pool=np.arange(20)
)
print("Interaction pairs:", interaction_pairs)
print("Gamma vector:", gamma_vec)
print("Interaction effects:", eta_int)

Interaction pairs: [(0, 1), (12, 16), (5, 6)]
Gamma vector: [0.46538668 1.67829799 1.21892946]
Interaction effects: [ 1.51044537 -2.00766932 -1.72595242  0.6786796   1.84498455 -0.65993255
  2.41759964 -5.15549961 -3.16320278  0.60732284]


In [15]:
def _calibrate_intercept(eta, ards_rate, iters=50, step=2.0):
    b0 = 0.0
    for _ in range(iters):
        probs = expit(b0 + eta)
        diff = probs.mean() - ards_rate
        b0 -= diff * step
    probs = expit(b0 + eta)
    return b0, probs

b0, probs = _calibrate_intercept(eta_main + eta_int, ards_rate=0.2)
print("Calibrated intercept:", b0)
print("Resulting probabilities mean:", probs.mean())

Calibrated intercept: -1.6604397235870885
Resulting probabilities mean: 0.2000079689084438


In [16]:
def _summ(x):
    return dict(
        shape=tuple(x.shape),
        mean=float(np.mean(x)),
        std=float(np.std(x)),
        min=float(np.min(x)),
        max=float(np.max(x)),
    )

_summ(proteins)

{'shape': (10, 20),
 'mean': 0.14425961055627615,
 'std': 1.2672982973450693,
 'min': -3.37492208981571,
 'max': 3.349238291288273}

### Main only

In [3]:
X_sim, y_sim, truth = simulate_patients(n=2000, p=500, effect_mode="main", n_interactions=0)

In [22]:
print(truth["signal_proteins"])
print(truth["interaction_pair_names"])
print(truth["block_id"][165])

['prot_0166', 'prot_0286', 'prot_0191', 'prot_0200', 'prot_0105', 'prot_0459', 'prot_0330', 'prot_0310']
[]
6


In [6]:
protein_cols = [f"seq.{i}" for i in range(X_sim.shape[1])]

df_sim = X_sim.copy()
df_sim.columns = protein_cols          # <- detta är nyckeln
df_sim["ards"] = np.asarray(y_sim).astype(int)

print(truth['block_id'])

[9 2 7 1 6 7 2 2 5 4 4 6 3 0 9 1 7 1 7 2 9 8 7 0 6 9 8 4 0 1 2 7 1 2 4 2 4
 6 1 3 7 1 8 8 6 2 7 8 0 6 4 6 1 0 4 8 0 1 4 8 2 6 8 9 8 4 8 5 8 0 3 4 6 8
 3 8 6 5 7 8 2 4 7 6 4 5 8 8 0 4 4 0 9 4 2 9 4 3 5 1 5 9 1 6 7 2 2 6 0 4 8
 9 0 5 0 6 7 4 3 8 6 5 9 9 9 3 5 0 5 9 5 2 4 1 8 6 8 0 6 4 8 9 3 5 5 7 3 2
 9 6 1 3 0 7 5 7 0 3 9 7 4 0 0 6 3 6 1 1 5 4 2 5 9 6 8 3 4 6 6 4 3 7 5 1 9
 9 2 3 5 8 2 2 8 7 1 4 2 2 4 9 6 4 5 9 0 9 5 9 8 1 6 2 6 4 3 0 9 3 7 7 8 2
 8 4 7 6 2 6 2 1 6 2 7 1 7 0 2 0 7 9 1 3 0 4 8 3 3 0 5 0 0 9 0 3 4 0 5 3 0
 7 3 8 2 3 8 3 9 9 5 4 6 6 6 9 5 9 0 4 4 1 7 3 1 5 5 7 7 7 1 0 2 2 5 5 2 2
 7 9 6 4 0 5 9 3 7 5 1 2 1 9 4 1 4 5 2 8 0 2 8 1 6 1 1 3 3 6 6 0 8 6 7 3 0
 5 7 6 3 5 1 6 9 6 8 8 2 6 6 4 0 0 1 9 5 9 9 9 7 7 7 8 3 5 7 2 7 6 7 1 7 8
 8 2 3 7 0 2 5 3 1 5 4 8 0 3 4 3 3 6 3 9 6 0 4 3 2 5 9 7 9 3 8 6 1 9 3 9 4
 5 3 2 5 9 4 0 4 7 8 1 0 3 7 3 5 0 1 8 8 6 2 1 8 0 6 0 0 9 8 4 8 4 8 4 8 5
 1 8 2 4 9 2 6 9 1 1 5 9 9 6 0 4 7 7 5 5 1 9 3 1 2 1 5 5 7 8 3 1 0 7 1 2 5
 7 2 4 1 3 1 3 1 7 2 2 5 

### Interactions only
Här är interaction_from_signals=True meningslöst eftersom det inte finns “signalproteiner” i den moden; därför faller koden tillbaka till att dra från alla proteiner.

In [ ]:
X_sim, y_sim, truth = simulate_patients(effect_mode="interaction_only", n_interactions=2, interaction_from_signals=False)

### Interactions + main

In [ ]:
X_sim, y_sim, truth = simulate_patients(effect_mode="main_plus_interaction", n_interactions=2, interaction_from_signals=True)

### Test batteri

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

def quick_checks():
    # 1) reproducibility
    X1, y1, t1 = simulate_patients(seed=123, effect_mode="main", n_interactions=0)
    X2, y2, t2 = simulate_patients(seed=123, effect_mode="main", n_interactions=0)
    assert np.allclose(X1.values, X2.values), "X not reproducible for same seed"
    assert np.array_equal(y1.values, y2.values), "y not reproducible for same seed"
    assert np.array_equal(t1["signal_idx"], t2["signal_idx"]), "signal_idx not reproducible"

    # 2) shapes + finite
    n, p = 250, 300
    assert X1.shape == (n, p)
    assert y1.shape == (n,)
    assert np.isfinite(X1.values).all()

    # 3) prevalence close to target
    target = 0.2
    prev = y1.mean()
    assert abs(prev - target) < 0.04, f"Prevalence off: got {prev:.3f}, target {target:.3f}"

    # 4) truth consistency
    X_main, y_main, tr_main = simulate_patients(seed=1, effect_mode="main", n_interactions=0)
    assert tr_main["signal_idx"] is not None and tr_main["beta_vec"] is not None
    assert len(tr_main["interaction_pairs"]) == 0

    X_int, y_int, tr_int = simulate_patients(seed=1, effect_mode="interaction_only", n_interactions=2)
    assert tr_int["signal_idx"] is None and tr_int["beta_vec"] is None
    assert len(tr_int["interaction_pairs"]) == 2 and tr_int["gamma_vec"] is not None

    X_both, y_both, tr_both = simulate_patients(seed=1, effect_mode="main_plus_interaction", n_interactions=2)
    assert tr_both["signal_idx"] is not None and tr_both["beta_vec"] is not None
    assert len(tr_both["interaction_pairs"]) == 2 and tr_both["gamma_vec"] is not None

    # 5) interaction pairs validity
    pairs = tr_both["interaction_pairs"]
    assert len(pairs) == len(set(pairs)), "Duplicate interaction pairs"
    for (j1, j2) in pairs:
        assert 0 <= j1 < p and 0 <= j2 < p and j1 != j2

    # 6) no-signal AUC ~ 0.5 (main)
    X0, y0, tr0 = simulate_patients(seed=2, effect_mode="main", n_interactions=0, beta_signal=0.0)
    Xtr, Xte, ytr, yte = train_test_split(X0, y0, test_size=0.3, random_state=0, stratify=y0)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(Xtr, ytr)
    auc0 = roc_auc_score(yte, clf.predict_proba(Xte)[:, 1])
    print("AUC (no main signal) =", round(auc0, 3))

    # 7) strong-signal AUC higher (main)
    Xs, ys, trs = simulate_patients(seed=2, effect_mode="main", n_interactions=0, beta_signal=2.0)
    Xtr, Xte, ytr, yte = train_test_split(Xs, ys, test_size=0.3, random_state=0, stratify=ys)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(Xtr, ytr)
    aucs = roc_auc_score(yte, clf.predict_proba(Xte)[:, 1])
    print("AUC (strong main signal) =", round(aucs, 3))

    # 8) block correlation sanity (rough check)
    # pick two proteins same block and two proteins different blocks
    block = tr_main["block_id"]
    j_same = None
    for j in range(p):
        k = block[j]
        cand = np.where(block == k)[0]
        if len(cand) >= 2:
            j_same = (cand[0], cand[1])
            break
    j_diff = (0, np.where(block != block[0])[0][0])

    corr_same = np.corrcoef(X_main.values[:, j_same[0]], X_main.values[:, j_same[1]])[0,1]
    corr_diff = np.corrcoef(X_main.values[:, j_diff[0]], X_main.values[:, j_diff[1]])[0,1]
    print("Corr same block =", round(corr_same, 3), "| Corr different blocks =", round(corr_diff, 3))

    print("✅ quick_checks passed")

quick_checks()

# Testa t-tester

1. Main only, låg korrelation, låg noise

effect_mode="main"

n_interactions=0

noise_sd=0.2–0.4

gärna block_strength=0.0 först (oberoende proteiner)
Förväntan: t-test ska hitta många av signalerna.

2. Main only, med korrelation

block_strength=1.0
Förväntan: t-test hittar ofta några signaler + “hitchhikers” i samma block.

Interaction only

effect_mode="interaction_only"
Förväntan: univariat t-test missar ofta (för att marginaleffekt kan vara nära 0). Bra “expected failure”.

3. Main + interaction

blandad realism.

Det här är exakt så man bygger ett testbatteri.